# Testing Agents Without a Model

**What you will build:** a test suite for an agent that runs in **milliseconds, offline, for free** — using PydanticAI's fake models (`TestModel`, `FunctionModel`), run-away protection (`UsageLimits`), and honest cost accounting. This is the chapter that turns "my agent seems to work" into engineering.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/15b_testing_agents.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — PydanticAI on OpenRouter. Run once.
!pip install -q "pydantic-ai-slim[openai]>=2.0,<3.0"   # verified against pydantic-ai 2.5 (2026)

import os, getpass
from pydantic_ai import Agent, ModelRetry, RunContext
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL_NAME = "meta-llama/llama-3.3-70b-instruct"
model = OpenAIChatModel(
    MODEL_NAME,
    provider=OpenAIProvider(base_url="https://openrouter.ai/api/v1",
                            api_key=os.environ["OPENROUTER_API_KEY"]),
)
print("Ready:", MODEL_NAME)

## Why testing agents is its own problem

Try to write an ordinary unit test for an agent and three walls appear at once: the output is **non-deterministic** (0.0 — the same input varies), every test run **costs tokens and seconds**, and the test **fails when the network hiccups** even though your code is fine. Teams respond by not testing — and then every refactor is a leap of faith.

The way out is to separate two questions that look similar and aren't:

| Question | Layer | Tooling | Cost |
|---|---|---|---|
| "Is my *plumbing* right?" — tools registered, schemas correct, validators fire, deps flow, errors handled | **Unit tests** | fake models: `TestModel`, `FunctionModel` | free, offline, milliseconds |
| "Is my agent any *good*?" — right answers, right tool choices, quality over a dataset | **Evals** | real models + `pydantic-evals` | tokens, seconds per case |

This notebook is the first row. The second row is the next chapter (1.6) — and it only pays off once the first row is solid, because an eval that fails on a plumbing bug wastes real tokens telling you something a millisecond test could have.

An agent under test, with the pieces from the last three chapters — a tool, typed output, and an argument guard:

In [ ]:
from dataclasses import dataclass
from typing import Literal
from pydantic import BaseModel


@dataclass
class Deps:
    plans: dict            # customer plan database


class Reply(BaseModel):
    message: str
    escalate: bool


support = Agent(
    model,
    deps_type=Deps,
    output_type=Reply,
    system_prompt=("You are a support agent. Look up the customer's plan when relevant. "
                   "Escalate anything about refunds."),
)

@support.tool
def get_plan(ctx: RunContext[Deps], customer: str) -> str:
    """Look up which plan a customer is on. customer is a lowercase username, e.g. 'alex'."""
    if customer != customer.lower():
        raise ModelRetry(f"{customer!r} is not a lowercase username. Try {customer.lower()!r}.")
    plan = ctx.deps.plans.get(customer)
    if plan is None:
        return f"No customer named {customer}."
    return f"{customer} is on the {plan} plan."

## `TestModel`: the smoke test

`TestModel` is a fake model that never touches the network. By default it calls **every tool once** with schema-valid arguments, then produces an output that satisfies your `output_type`. That default is exactly what a smoke test wants: *if anything is miswired — a broken schema, a tool that crashes on valid input, an impossible output type — this run explodes, in milliseconds, for free.*

The idiomatic way in is `agent.override(...)`, a context manager that swaps the model inside the block — your production wiring stays untouched:

In [ ]:
from pydantic_ai.models.test import TestModel

test_model = TestModel()

with support.override(model=test_model):
    result = await support.run("Which plan is alex on?", deps=Deps(plans={"alex": "Pro"}))

print("output:", result.output)
print("type:  ", type(result.output).__name__)

called = [p.tool_name for m in result.all_messages()
          for p in m.parts if p.part_kind == "tool-call"]
print("tool calls:", called)          # note: includes 'final_result' — the output tool (see below)
assert "get_plan" in called, called
assert isinstance(result.output, Reply)
print("assertions passed — plumbing is sound")

Read what just got verified without spending a token: the tool schema is generatable, `get_plan` executes on schema-valid input, deps flow through `RunContext`, and the `Reply` model is satisfiable. The *content* is nonsense (`TestModel` invents placeholder values — that's not its job); the *machinery* is proven.

One more `TestModel` service you met in 1.2: `test_model.last_model_request_parameters` records the exact tool definitions the model would receive — assert on it and a colleague can't silently break a docstring your agent depends on:

In [ ]:
tools_seen = {t.name: t.description for t in test_model.last_model_request_parameters.function_tools}
assert "get_plan" in tools_seen
assert "lowercase username" in tools_seen["get_plan"]     # the description IS the interface (1.2)
print("tool contract locked:", tools_seen)

## `FunctionModel`: script the model, test the machinery

`TestModel` picks its own (generic) behavior. To test a *specific* path — "what happens when the model passes a bad argument?" — you want to play the model's role yourself. `FunctionModel` takes a function that receives the messages and returns the next `ModelResponse`: the model as a puppet.

Here's the test that actually matters for our agent: **does the `ModelRetry` guard work?** We script a model that first calls the tool *wrongly*, then corrects itself only if it received the retry feedback:

In [ ]:
from pydantic_ai.models.function import FunctionModel, AgentInfo
from pydantic_ai.messages import ModelMessage, ModelResponse, TextPart, ToolCallPart


def stubborn_then_corrected(messages: list[ModelMessage], info: AgentInfo) -> ModelResponse:
    """Turn 1: call get_plan with an ILLEGAL argument (uppercase).
    Turn 2+: if the retry feedback arrived, call it correctly.
    Finally: produce the typed output."""
    last_parts = messages[-1].parts
    if any(p.part_kind == "retry-prompt" for p in last_parts):
        return ModelResponse(parts=[ToolCallPart(tool_name="get_plan",
                                                 args={"customer": "alex"})])
    if any(p.part_kind == "tool-return" for p in last_parts):
        return ModelResponse(parts=[ToolCallPart(
            tool_name="final_result",
            args={"message": "alex is on the Pro plan", "escalate": False})])
    return ModelResponse(parts=[ToolCallPart(tool_name="get_plan",
                                             args={"customer": "ALEX"})])   # deliberately wrong


with support.override(model=FunctionModel(stubborn_then_corrected)):
    result = await support.run("Which plan is alex on?", deps=Deps(plans={"alex": "Pro"}))

kinds = [p.part_kind for m in result.all_messages() for p in m.parts]
assert "retry-prompt" in kinds, "the ModelRetry guard never fired!"
assert result.output.escalate is False
print("retry path verified:", " -> ".join(kinds))

Look at the printed part-kind trail: `tool-call` (bad) → `retry-prompt` (your guard's message) → `tool-call` (corrected) → `tool-return` → `tool-call` (`final_result`) → `tool-return`. You just unit-tested a *conversation shape* — the self-correction loop — deterministically, offline, in milliseconds. This is the test you could never write reliably against a real model, because a good model might pass a correct argument first try and your guard would go silently untested.

(One implementation note: with an `output_type`, the final structured answer travels as a tool call named `final_result` — that's how PydanticAI transports typed output on the wire, and why the script's last step calls it.)

## `UsageLimits`: the typed `max_turns`

Back in 0.3 your loop had a `max_turns` valve, and 1.0's comparison table promised the framework handles it. Here's where: `UsageLimits`, enforced per run. Two limits matter — **requests** (model calls: the loop valve) and **tokens** (the budget valve):

In [ ]:
from pydantic_ai.usage import UsageLimits
from pydantic_ai.exceptions import UsageLimitExceeded

looper = Agent(model, system_prompt="You are a helpful assistant. Use tools when they help.")

@looper.tool_plain
def try_again(task: str) -> str:
    """Attempt a task. May need several attempts."""
    return "Not done yet — call try_again once more."     # a tool that invites an infinite loop

try:
    await looper.run("Use try_again until the task is finished.",
                     usage_limits=UsageLimits(request_limit=3))
except UsageLimitExceeded as e:
    print("stopped by the valve:", e)

Without the limit, that tool is an infinite loop billed by the turn — with it, the run raises `UsageLimitExceeded` after three model calls, cleanly, before the bill grows. Two habits worth adopting now: production runs always carry a `request_limit` (a generous one — it's a fuse, not a feature), and cost-sensitive paths add `total_tokens_limit=`. Note the exception is *not* `UnexpectedModelBehavior` (1.5): that one means "the model kept failing validation"; this one means "the run was working but exceeded its budget". Catching them separately tells you which problem you have.

## What a run costs, in money

The `usage` attribute has been on every result since 1.1; turning it into euros is one function, and it changes conversations with stakeholders ("each support reply costs ~$0.002" lands differently than "it uses about 900 tokens"). Prices live on [openrouter.ai/models](https://openrouter.ai/models) per million tokens:

In [ ]:
# Prices for the course model, per MILLION tokens (check openrouter.ai/models for current):
PRICE_IN, PRICE_OUT = 0.10, 0.25          # USD/Mtok — example figures, verify before quoting

def run_cost(usage) -> float:
    return usage.input_tokens * PRICE_IN / 1e6 + usage.output_tokens * PRICE_OUT / 1e6

result = await support.run("Which plan is alex on?", deps=Deps(plans={"alex": "Pro"}))   # real model this time
u = result.usage
print(f"{u.input_tokens} in + {u.output_tokens} out tokens  ->  ${run_cost(u):.6f}")
print(f"1,000 such requests/day -> ${run_cost(u) * 1000 * 30:.2f}/month")

::::{dropdown} 🔧 Under the hood: shipping this as a real test suite
:color: info

In a project (not a notebook), these become ordinary pytest functions. Two idioms complete the picture:

```python
# test_support_agent.py
import pytest
from pydantic_ai import models
from pydantic_ai.models.test import TestModel

models.ALLOW_MODEL_REQUESTS = False        # the safety catch: ANY accidental real
                                           # model call in the test suite now raises

@pytest.mark.anyio
async def test_smoke():
    with support.override(model=TestModel()):
        result = await support.run("Which plan is alex on?", deps=Deps(plans={"alex": "Pro"}))
    assert isinstance(result.output, Reply)
```

`ALLOW_MODEL_REQUESTS = False` is the line that makes the suite trustworthy in CI: no key, no network, no surprise invoices — a forgotten `override` fails loudly instead of silently costing money. This split (fast fake-model tests in CI on every commit; evals with real models run deliberately, on a schedule or before releases) is the norm the best teams converge on.
::::

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Test the refund cap.** Take the `refund` tool from 1.5 and write a `FunctionModel` test where the scripted model first requests a $500 refund on order A-1002 ($349). **Done when:** your assertions prove the retry fired and no refund above the order total ever returned from the tool.
2. **Find the floor.** For the two-tool stock question from 1.2 ("10 shares of AAPL in euros"), find the minimal `request_limit` that lets it finish. **Done when:** you can explain the number in terms of the loop's turns (hint: each batch of tool calls needs a model call, plus one to answer).
3. **Price an experiment.** Compute what the vague-vs-clear measurement in 1.2 (10 runs) actually cost, using `run_cost`. **Done when:** you have a dollar figure — and an opinion about whether measuring reliability was worth it (it was).
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**1 — Refund cap test:**

```python
def demands_500(messages, info):
    last = messages[-1].parts
    if any(p.part_kind == "retry-prompt" for p in last):
        return ModelResponse(parts=[ToolCallPart(tool_name="refund",
                                                 args={"order_id": "A-1002", "amount": 349.0})])
    if any(p.part_kind == "tool-return" for p in last):
        return ModelResponse(parts=[TextPart("Refunded the maximum: $349.")])
    return ModelResponse(parts=[ToolCallPart(tool_name="refund",
                                             args={"order_id": "A-1002", "amount": 500.0})])

with support_15.override(model=FunctionModel(demands_500)):     # the 1.5 agent with the refund tool
    result = await support_15.run("Refund $500 on A-1002.")

kinds = [p.part_kind for m in result.all_messages() for p in m.parts]
assert "retry-prompt" in kinds
returns = [str(p.content) for m in result.all_messages()
           for p in m.parts if p.part_kind == "tool-return"]
assert all("500" not in r for r in returns)      # no $500 refund ever executed
```

The guard is now *pinned by a test*: if someone later deletes the amount check, CI goes red before a customer gets a $500 refund on a $349 chair.

**2 — The floor is 3**: call 1 (model requests `get_stock_price`), call 2 (requests `convert_currency` after seeing the price), call 3 (writes the answer). If your model batches both tools in one turn (parallel tool calls, 0.3), the floor drops to 2. The exercise's real yield: you now read `request_limit` in loop-turns, not as a magic number.

**3 —** With the example prices, 10 short runs land around a few hundredths of a cent to a cent total. The point of the exercise is the reflex: *experiments have a price tag, and it's almost always worth paying* — the cost of NOT knowing your tool-call reliability is a production incident.
::::

::::{dropdown} 🔧 Common issues
:color: info

| Symptom | Likely cause | Fix |
|---|---|---|
| `TestModel` output looks like gibberish | Working as designed — it invents schema-valid placeholders | Assert on *structure* (types, tool calls, part kinds), never on content |
| A `FunctionModel` test hangs or loops | Your script never returns a final answer (text or `final_result`), so the loop keeps going | Make the last branch unconditional; add `usage_limits=UsageLimits(request_limit=5)` to the test run as a belt |
| `final_result` tool not found in your script | The agent has no `output_type`, so there's no output tool | Plain-text agents end with `ModelResponse(parts=[TextPart("...")])` instead |
| Real tokens spent during tests | A code path built its own `Agent(model)` bypassing `override` | Set `models.ALLOW_MODEL_REQUESTS = False` at the top of the suite — it turns silent spending into a loud error |
| `UsageLimitExceeded` vs `UnexpectedModelBehavior` confusion | Both end a run with an exception | Budget exceeded vs validation exhausted — catch them separately, they need different fixes |
::::

**What's next:** the plumbing is tested; now the *quality*. In **1.6** we build **evals** — measuring whether the agent gives good answers over a dataset of real cases, with pass rates instead of impressions. Everything here carries over: cheap fake-model tests catch regressions per commit; evals judge quality per release.